In [ ]:
%pip install -q databricks-feature-engineering

In [ ]:
%run ./_common

In [ ]:
from pyspark.sql.functions import col, monotonically_increasing_id

In [ ]:
# Create a delta table for diet features
@SWSAcademyHelper.add_init
def create_diet_features_table(self):
    
    spark.sql(f"USE CATALOG {SA.catalog_name}")
    spark.sql(f"USE SCHEMA {SA.schema_name}")

    # The lab adds a primary key to diet_features and creates diabetes_features,
    # so start from fresh tables on every attempt.
    for name in ("diet_features", "diabetes_features"):
        spark.sql(f"DROP TABLE IF EXISTS {SA.catalog_name}.{SA.schema_name}.{name}")

    # Set the path of the dataset
    shared_volume_name = 'cdc-diabetes' # From Marketplace
    csv_name = 'diabetes_binary_5050split_BRFSS2015' # CSV file name
    dataset_path = f"{SA.paths.datasets.cdc_diabetes}/{shared_volume_name}/{csv_name}.csv" # Full path


    df = spark.read.csv(dataset_path, header="true", inferSchema="true", multiLine="true", escape='"')
    df = df.select("Fruits", "Veggies", "HvyAlcoholConsump", "Smoker")
    df = df.withColumn("UID", monotonically_increasing_id())

    table_name = f"{SA.catalog_name}.{SA.schema_name}.diet_features"
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [ ]:
# Initialize SWSAcademyHelper
SA = SWSAcademyHelper() 
SA.init()